# Day 05 – Tiny Gene Regulatory Network Peek

We build a quick correlation network so you can spot likely transcription-factor hubs. For production pipelines swap in pySCENIC, CellOracle, etc.


### What happens today?
1. Load annotated data.
2. Focus on a manageable set of genes (top 40) and compute pairwise Spearman correlations.
3. Turn strong correlations into a simple NetworkX graph to inspect hubs.


In [ ]:
# Install the needed libraries once (delete the # to run)
# %pip install --quiet scanpy scvi-tools scvelo gseapy networkx


### Step 1 – Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

try:
    import scvi
except ImportError:
    scvi = None
    print('⚠️ Install scvi-tools (pip install scvi-tools) to unlock the perturbation modeling demo.')

try:
    import gseapy as gp
except ImportError:
    gp = None
    print('⚠️ Install gseapy (pip install gseapy) to run the GSEA step.')

try:
    import scvelo as scv
except ImportError:
    scv = None
    print('⚠️ Install scvelo (pip install scvelo) to run the RNA velocity step.')

import networkx as nx

sc.settings.verbosity = 0
sc.set_figure_params(dpi=100)


### Step 2 – Load annotated data

In [ ]:
SHARED_DIR = Path('..') / 'shared_data'
SHARED_DIR.mkdir(parents=True, exist_ok=True)
day2_file = SHARED_DIR / 'day02_annotated.h5ad'

print('Looking for Day 02 annotated data at', day2_file)

def build_day2_from_scratch():
    """Create the Day 02 dataset (clean + cluster + annotate) on the fly."""
    data = sc.datasets.pbmc3k()
    data.var_names_make_unique()
    data.layers['counts'] = data.X.copy()
    data.obs['source_dataset'] = 'pbmc3k'
    data.raw = data

    sc.pp.filter_cells(data, min_genes=200)
    sc.pp.filter_genes(data, min_cells=3)
    data.var['mt'] = data.var_names.str.upper().str.startswith('MT-')
    sc.pp.calculate_qc_metrics(data, qc_vars=['mt'], inplace=True)
    data = data[data.obs['pct_counts_mt'] < 15, :]

    sc.pp.normalize_total(data, target_sum=1e4)
    sc.pp.log1p(data)
    sc.pp.highly_variable_genes(data, n_top_genes=2000, subset=True)
    sc.pp.scale(data, max_value=10)

    sc.tl.pca(data, n_comps=50)
    sc.pp.neighbors(data, n_neighbors=15)
    sc.tl.umap(data)

    sc.tl.leiden(data, resolution=0.5, key_added='leiden')
    marker_map = {
        '0': 'Naive T',
        '1': 'Memory T',
        '2': 'B cell',
        '3': 'NK',
        '4': 'Myeloid',
        '5': 'Plasma',
    }
    data.obs['cell_type'] = data.obs['leiden'].map(marker_map).fillna('Other')
    data.obs['batch'] = 'Batch_A'
    return data

if day2_file.exists():
    adata_from_day2 = sc.read(day2_file)
    print('✅ Loaded annotated data from Day 02 file.')
else:
    print('⚠️ Day 02 file not found. Re-running the quick Day 02 pipeline now...')
    adata_from_day2 = build_day2_from_scratch()


In [ ]:
adata_day5 = adata_from_day2.copy()


### Step 3 – Build a toy regulatory network

In [ ]:
key_genes = list(adata_day5.var_names[:40])
subset = adata_day5[:, key_genes].X
if hasattr(subset, 'toarray'):
    subset = subset.toarray()
expr_df = pd.DataFrame(subset, columns=key_genes)

corr = expr_df.corr(method='spearman')
threshold = 0.45
G = nx.Graph()
G.add_nodes_from(key_genes)
for i, gene_i in enumerate(key_genes):
    for gene_j in key_genes[i + 1:]:
        weight = corr.loc[gene_i, gene_j]
        if abs(weight) >= threshold:
            G.add_edge(gene_i, gene_j, weight=float(weight))

print(f'Nodes: {G.number_of_nodes()} | Edges (|r| >= {threshold}): {G.number_of_edges()}')
degree_series = pd.Series(dict(G.degree())).sort_values(ascending=False)
display(degree_series.head(10))
